# Regressione logistica

In [9]:
import pandas as pd
import numpy as np


from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, make_scorer, classification_report

from joblib import Parallel, delayed
import math
import re
from tabulate import tabulate
from pathlib import Path
import warnings 
from sklearn.exceptions import ConvergenceWarning
import time
from datetime import timedelta
# Nascondo i warning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare treining
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training

In [10]:
def training(file_path, csv_name):
      

    df = pd.read_csv(file_path)

    # Definizione dei target originali 
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]', 'HER2 [SII]']

    # Rimozione delle righe che non hanno i valori target 
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo i valori continui in classi binarie (0/1) basate su soglie cliniche.
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    df_validi['HER2_class'] = (df_validi['HER2 [SII]'] >= 3).astype(int)

    # Lista dei nuovi nomi delle colonne target binarizzate
    final_target_list = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

    # Definizione delle colonne da escludere dalle features:
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    
    # Rimuovo le colonne inutili
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # Creo target e gruppi per la cross-validation
    target = df_validi[final_target_list]
    groups = df_validi['Patient ID'] # Importante: raggruppare per paziente evita data leakage tra train/test

    # Riempo le celle vuote con la media
    features = features.fillna(features.mean())
    
    # Vado a rimuvoere i carattei speciali
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]

    # Configuro la cross-validation a 5 fold
    cv = GroupKFold(n_splits=5)

    # Configurazione  della Logistic Regression
    base_model = LogisticRegression(
        random_state=42, 
        n_jobs=1,               
        class_weight='balanced',              
        solver='saga',         
        max_iter=2000          
    )
    
    # MultiOutputClassifier mi permette di predire 4 target contemporaneamente 
    multi_output_model = MultiOutputClassifier(base_model)


    # Definisco gli iperparametri
    iperparametri = {
        'estimator__C': [0.01, 0.1, 1, 10],         # Inversa della forza di regolarizzazione (valori bassi = più regolarizzazione)
        'estimator__penalty': ['l2', 'elasticnet'], # l2 (Ridge) standard, elasticnet (mix L1/L2) ottima per feature selection
        'estimator__l1_ratio': [0.5],               # Usato solo se penalty='elasticnet'. 0.5 significa mix bilanciato.
    }

    # Funzione per calcolare lo score medio su tutti i 4 target
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        # Calcola F1 macro per ogni target 
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0))
        # Ritorna la media degli F1 score dei 4 target
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    # Calcolo combinazioni totali solo per log informativo
    total_combinations = math.prod(len(v) for v in iperparametri.values())
    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name}")

    # Eseguo la GridSearchCV
    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=cv,                  # Uso la GroupKFold definita prima
        scoring=scorer,         # Uso lo scorer custom
        n_jobs=-1,              # Uso tutti i core della CPU
        verbose=1,              # Stampa progresso base
        refit=True,             # Rifitta il modello migliore su tutti i dati alla fine
        error_score='raise'     # Alza eccezione se il fitting fallisce (utile per debug)
    )

    # Avvio ricerca 
    grid_search.fit(features, target, groups=groups)

    # Analizzo i risultati
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    fold_reports = []

    # Vado a rimuovere "estimator__" 
    clean_best_params = {k.replace('estimator__', ''): v for k, v in best_params.items()}
    
    # Compongo insieme tutti i parametri usati
    final_params = {
        'random_state': 42,
        'n_jobs': 1,
        'class_weight': 'balanced',
        'solver': 'saga',
        'max_iter': 2000,
        **clean_best_params # Unpacking dei migliori parametri trovati
    }

    # Creo il report a mano
    for train_idx, test_idx in cv.split(features, target, groups):
        # Splitting manuale usando gli indici del GroupKFold
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        # Creazione e training di un clone del modello migliore
        model_clone = MultiOutputClassifier(LogisticRegression(**final_params))
        model_clone.fit(X_train, y_train)
        
        # Predizione sul test set del fold corrente
        y_pred = model_clone.predict(X_test)

        # Generazione report per ogni target separatamente
        report_dict = {}
        for i, col in enumerate(final_target_list):
            # classification_report restituisce precision, recall, f1 per classe 0 e 1
            report_dict[col] = classification_report(
                y_test.iloc[:, i],
                y_pred[:, i],
                output_dict=True,
                zero_division=0
            )
        fold_reports.append(report_dict)

    
    final_result = [{
        **clean_best_params,
        'mean_score': best_score, # Score medio della GridSearch
        # Deviazione standard dei risultati della CV
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports # Report dettagliati fold per fold
    }]

    return final_result


# Stampo i risultati in un formato leggibile

In [11]:
def print_grid_search_results(results_per_dataset):
    
    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI (Logistic Regression)")
    print("=" * 80)

    # Lista per raccogliere i dati riassuntivi per la tabella finale di confronto
    summary_data = []

    # Itero su ogni dataset presente nel dizionario dei risultati
    for name, metrics_list in results_per_dataset.items():
        
        # Assumo che in metrics_list[0] ci sia il miglior risultato
        best_result = metrics_list[0]

        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")

        # Stampo gli iperparametri ottimali
        print("Iperparametri Ottimali:")
        
        possible_params = [
            ('C (Inverse Reg)', 'C'),       
            ('Penalty', 'penalty'),         
            ('L1 Ratio', 'l1_ratio'),       
            ('Solver', 'solver')            
        ]
        
       
        params_table = []
        for label, key in possible_params:
            val = best_result.get(key)
            # Includo il parametro solo se esiste 
            if val is not None:
                 params_table.append([label, val])

        # stampo col comando "tabulate", cosi lo formatto a modo di tabella
        print(tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))


        print("\n Metriche di Classificazione per Target (Dettaglio):\n")
        
        target_names = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

        # Controllo se esistono report dettagliati dei fold
        if best_result.get('fold_reports'):

            first_fold_report = best_result['fold_reports'][0]

            for target_name in target_names:
                # Salta se il target non è presente nel report
                if target_name not in first_fold_report:
                    continue

                current_target_report = first_fold_report[target_name]

                rows = []
                # Filtra solo le classi '0' e '1' per evitare le medie macro nel dettaglio
                classes = [c for c in ['0', '1'] if c in current_target_report]

                for cls in classes:
                    rows.append([
                        f"Classe {cls}",
                        f"{current_target_report[cls]['precision']:.3f}",
                        f"{current_target_report[cls]['recall']:.3f}",
                        f"{current_target_report[cls]['f1-score']:.3f}",
                        int(current_target_report[cls]['support'])
                    ])

                # Stampa tabella per il target corrente
                print(f"  Target: {target_name}")
                print(tabulate(rows, headers=['', 'Precision', 'Recall', 'F1-score', 'Support'],
                             tablefmt='simple', colalign=('left', 'center', 'center', 'center', 'center')))
                print()
        else:
            print("Nessun report dettagliato disponibile.")

        # Raccolta dati per il riepilogo finale
        summary_data.append([
            name,
            f"{best_result['mean_score']:.3f}",
            f"{best_result['std_score']:.3f}",
            best_result.get('C'),
            best_result.get('penalty'),
            best_result.get('l1_ratio') if best_result.get('penalty') == 'elasticnet' else '-'
        ])


    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")

    # Ordino il dataset, in base alla f1_score dal migliore al peggiore
    summary_data.sort(key=lambda x: float(x[1]), reverse=True)

    print(tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std Dev', 'C', 'Penalty', 'L1 Ratio'],
                   tablefmt='grid',
                   floatfmt=('.3f', '.3f', '.3f', '.0f', '.0f', '.0f')))

# Eseguo il tutto

In [12]:
start_time = time.time()


# Eseguo il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)



end_time = time.time()
# Calcolo il tempo impiegato
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



Inizio Grid Search (8 combinazioni) per: t2_medsam
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when


Inizio Grid Search (8 combinazioni) per: t2_preprocessed
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when


Inizio Grid Search (8 combinazioni) per: t2_original
Fitting 5 folds for each of 8 candidates, totalling 40 fits


...


Inizio Grid Search (8 combinazioni) per: preprocessed_dynamic
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when


Inizio Grid Search (8 combinazioni) per: original_dynamic
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when


                         RIEPILOGO DEI MIGLIORI RISULTATI (Logistic Regression)

────────────────────────────────────────────────────────────────────────────────
 Dataset: t2_medsam
────────────────────────────────────────────────────────────────────────────────

 Performance: F1-score = 0.294 ± 0.117

Iperparametri Ottimali:
Parametro        Valore
---------------  --------
C (Inverse Reg)  0.01
Penalty          l2
L1 Ratio         0.5

 Metriche di Classificazione per Target (Dettaglio):

  Target: PR_class
           Precision    Recall    F1-score    Support
--------  -----------  --------  ----------  ---------
Classe 0     0.25         1         0.4          3
Classe 1       0          0          0           9

  Target: ER_class
           Precision    Recall    F1-score    Support
--------  -----------  --------  ----------  ---------
Classe 0     0.083        1        0.154         1
Classe 1       0          0          0          11

  Target: KI67_class
           Precision